In [ ]:
import os
import re, json, time
import unicodedata
import hashlib
import pickle
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple, Union
from datetime import datetime, timedelta, date

import pandas as pd
import tiktoken
from tqdm import tqdm

# Add these imports at the top
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import asyncio
import openai
import aiofiles
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache

from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.prompts import SystemMessagePromptTemplate

from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI


TRANSCRIPT_ROOT = Path("Transcripts_Deepgram_nova-2_order by patient copy")
PREVREPORTS_DIR = Path("PrevReports copy")
PATIENT_PROFILE_CSV = Path("patient_profiles copy.csv")

FORM_ROOT = Path("data/Visiting Form/Visiting Form - Order by Patient")  # <- set to your forms root


OUT_DIR = Path("singlePipeline_nowindow")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Config
# =========================

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

MODEL_EXTRACT   = "gpt-4o"        # fast, good enough with regex-first
MODEL_KNOWLEDGE = "gpt-4o-mini"   # cheap background notes
MODEL_SUMMARY   = "gpt-4.1"

FIELDS = ["体温","血圧","脈拍","呼吸","SPO2","排便","食欲"]

# =========================
# Performance Optimization Config
# =========================
MAX_CONCURRENT_FORMS = 3  # Concurrent form processing
MAX_TRANSCRIPT_TOKENS = 8000  # Smart transcript trimming
CACHE_ENABLED = True
EMBEDDING_CACHE_SIZE = 1000
SIMILARITY_THRESHOLD = 0.7  # RAG retrieval threshold

# Persistent cache directories
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
EXTRACTION_CACHE_FILE = CACHE_DIR / "extraction_cache.pkl"
EMBEDDING_CACHE_FILE = CACHE_DIR / "embedding_cache.pkl"
KNOWLEDGE_CACHE_FILE = CACHE_DIR / "knowledge_cache.pkl"

# Initialize caches
EXTRACTION_CACHE = {}
EMBEDDING_CACHE = {}
KNOWLEDGE_CACHE = {}

# Load persistent caches
def load_cache(cache_file: Path) -> dict:
    if cache_file.exists():
        try:
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Warning: Could not load cache {cache_file}: {e}")
    return {}

def save_cache(cache: dict, cache_file: Path):
    try:
        with open(cache_file, 'wb') as f:
            pickle.dump(cache, f)
    except Exception as e:
        print(f"Warning: Could not save cache {cache_file}: {e}")

# Load caches on startup
if CACHE_ENABLED:
    EXTRACTION_CACHE = load_cache(EXTRACTION_CACHE_FILE)
    EMBEDDING_CACHE = load_cache(EMBEDDING_CACHE_FILE)
    KNOWLEDGE_CACHE = load_cache(KNOWLEDGE_CACHE_FILE)
    print(f"Loaded caches: {len(EXTRACTION_CACHE)} extractions, {len(EMBEDDING_CACHE)} embeddings, {len(KNOWLEDGE_CACHE)} knowledge")

@lru_cache(maxsize=128)
def get_tokenizer(model_name: str):
    """Cached tokenizer for better performance"""
    if "gpt-4" in model_name:
        return tiktoken.encoding_for_model("gpt-4")
    elif "gpt-3.5" in model_name:
        return tiktoken.encoding_for_model("gpt-3.5-turbo")
    else:
        return tiktoken.get_encoding("cl100k_base")

# Keep original LLM clients for compatibility but add async alternatives
llm_extract = ChatOpenAI(
    model=MODEL_EXTRACT,
    temperature=0.0,
    openai_api_key=OPENAI_API_KEY,
)

llm_knowledge = ChatOpenAI(
    model=MODEL_KNOWLEDGE,
    temperature=0.0,
    openai_api_key=OPENAI_API_KEY,
)

llm_summarize = ChatOpenAI(
    model=MODEL_SUMMARY,
    temperature=0.3,       
    openai_api_key=OPENAI_API_KEY,
)

# Enhanced OpenAI client with JSON mode support and connection pooling
_client_pool = None

async def get_optimized_openai_client():
    """Get or create async OpenAI client with optimizations"""
    global _client_pool
    if _client_pool is None:
        _client_pool = openai.AsyncOpenAI(
            api_key=OPENAI_API_KEY,
            timeout=60.0,
            max_retries=3,
            # Connection pooling is handled internally by httpx
        )
    return _client_pool

async def extract_with_json_mode(transcript: str, patient_id: str) -> Dict[str, Any]:
    """Optimized Stage A extraction with enforced JSON mode and caching"""
    
    # Check cache first
    cache_key = hashlib.sha256(transcript.encode()).hexdigest()
    if CACHE_ENABLED and cache_key in EXTRACTION_CACHE:
        print(f"🎯 Cache HIT for extraction: {patient_id}")
        return EXTRACTION_CACHE[cache_key]
    
    # Trim transcript to reduce tokens
    trimmed_transcript = trim_transcript_smartly(transcript)
    
    # Optimized prompt for JSON mode (reduced tokens)
    json_prompt = f"""Extract health data from nursing visit transcript. Return ONLY valid JSON.

Fields: 体温, 血圧, 脈拍, 呼吸, SPO2, 排便, 食欲

Format per field:
- "value": exact measurement or null
- "source": "M" if mentioned, "" if not  
- "evidence": exact quote or ""

Rules:
- Only explicit measurements
- Blood pressure needs both values (e.g., "120/80")
- 排便/食欲: "有"/"無" only if clearly stated

Text: {trimmed_transcript}"""

    try:
        client = await get_optimized_openai_client()
        
        response = await client.chat.completions.create(
            model=MODEL_EXTRACT,
            messages=[{"role": "user", "content": json_prompt}],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        
        result = json.loads(response.choices[0].message.content)
        
        # Cache the result
        if CACHE_ENABLED:
            EXTRACTION_CACHE[cache_key] = result
            # Periodically save cache
            if len(EXTRACTION_CACHE) % 10 == 0:
                save_cache(EXTRACTION_CACHE, EXTRACTION_CACHE_FILE)
            
        print(f"✅ JSON extraction successful for {patient_id}")
        return result
        
    except Exception as e:
        print(f"❌ JSON extraction failed for {patient_id}: {e}")
        # Fallback to empty structure
        return {field: {"value": None, "source": "", "evidence": ""} for field in FIELDS}

# =========================
# Token & Cost metering
# =========================

PRICING = {
    "gpt-4.1":      {"in": 0.0020,  "out": 0.0080},   # $2.00 / $8.00 per 1M
    "gpt-4.1-mini": {"in": 0.0004,  "out": 0.0016},   # $0.40 / $1.60 per 1M
    "gpt-4o":       {"in": 0.0025,  "out": 0.0100},   # $2.50 / $10.00 per 1M
    "gpt-4o-mini":  {"in": 0.00015, "out": 0.0006},   # $0.15 / $0.60 per 1M
}


DEFAULT_PRICING = {"in": 0.01, "out": 0.03}  # fallback; adjust to your account if different

def count_tokens(text: str, model: str) -> int:
    """Optimized token counting with caching"""
    enc = get_tokenizer(model)
    return len(enc.encode(text or ""))

def trim_transcript_smartly(transcript: str, max_tokens: int = MAX_TRANSCRIPT_TOKENS) -> str:
    """Intelligently trim transcript while preserving key medical information"""
    if count_tokens(transcript, MODEL_EXTRACT) <= max_tokens:
        return transcript
    
    # Split into sections
    sections = transcript.split('\n\n')
    
    # Priority patterns for medical content
    priority_patterns = [
        (r'(体温|血圧|脈拍|呼吸|SPO2|排便|食欲)', 10),  # Vital signs - highest priority
        (r'(転倒|出血|発熱|痛み|呼吸苦|意識)', 9),  # Critical symptoms
        (r'(処置|ケア|観察|確認|清拭|交換)', 8),  # Care activities
        (r'(受診|検査|薬|内服|処方)', 7),  # Medical care
        (r'(状態|様子|表情|反応)', 6),  # General condition
        (r'(家族|介護|支援)', 5),  # Support system
    ]
    
    # Score sections by medical importance
    scored_sections = []
    for section in sections:
        score = 0
        for pattern, weight in priority_patterns:
            matches = len(re.findall(pattern, section))
            score += matches * weight
        
        # Boost score for longer sections (more context)
        score += len(section) / 100
        
        scored_sections.append((score, section))
    
    # Sort by score and rebuild transcript
    scored_sections.sort(key=lambda x: x[0], reverse=True)
    
    trimmed = ""
    for score, section in scored_sections:
        test_transcript = trimmed + "\n\n" + section if trimmed else section
        if count_tokens(test_transcript, MODEL_EXTRACT) > max_tokens:
            break
        trimmed = test_transcript
    
    return trimmed.strip() or transcript[:2000]  # Fallback to first 2000 chars

def estimate_cost(prompt_text: str, output_text: str, model: str) -> Tuple[int,int,float]:
    itok = count_tokens(prompt_text, model)
    otok = count_tokens(output_text, model)
    price = PRICING.get(model, DEFAULT_PRICING)
    cost = (itok/1000)*price["in"] + (otok/1000)*price["out"]
    return itok, otok, cost

class Meter:
    def __init__(self):
        self.records = []
    def add(self, step: str, prompt_text: str, output_text: str, seconds: float, model: str):
        it, ot, cost = estimate_cost(prompt_text, output_text, model)
        self.records.append({
            "step": step,
            "model": model,
            "input_tokens": it,
            "output_tokens": ot,
            "cost_usd": round(cost, 6),
            "seconds": round(seconds, 3),
        })
    def totals(self) -> Dict[str, Any]:
        it = sum(r["input_tokens"] for r in self.records)
        ot = sum(r["output_tokens"] for r in self.records)
        c  = sum(r["cost_usd"] for r in self.records)
        s  = sum(r["seconds"] for r in self.records)
        return {"input_tokens": it, "output_tokens": ot, "cost_usd": round(c,6), "seconds": round(s,3)}
    def as_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.records)


class OptimizedClinicalRAG:
    def __init__(self, api_key):
        import openai
        self.client = None  # Will be initialized async
        self.embedding_model = "text-embedding-3-small"
        self.knowledge_base = {}
        self.embedding_dim = 1536
        
    async def get_client(self):
        """Lazy async client initialization"""
        if self.client is None:
            self.client = await get_optimized_openai_client()
        return self.client
        
    async def encode_text_batch(self, texts: List[str]) -> List[np.ndarray]:
        """Batch encode texts for better API efficiency"""
        if not texts:
            return []
            
        # Filter empty texts
        valid_texts = [t for t in texts if t and t.strip()]
        if not valid_texts:
            return [np.zeros(self.embedding_dim) for _ in texts]
        
        # Check cache first
        cached_embeddings = []
        texts_to_encode = []
        cache_keys = []
        
        for text in valid_texts:
            cache_key = hashlib.sha256(text.encode()).hexdigest()
            cache_keys.append(cache_key)
            
            if CACHE_ENABLED and cache_key in EMBEDDING_CACHE:
                cached_embeddings.append(EMBEDDING_CACHE[cache_key])
            else:
                cached_embeddings.append(None)
                texts_to_encode.append(text)
        
        # Encode remaining texts in batch
        new_embeddings = {}
        if texts_to_encode:
            try:
                client = await self.get_client()
                response = await client.embeddings.create(
                    input=texts_to_encode,
                    model=self.embedding_model
                )
                
                for i, text in enumerate(texts_to_encode):
                    embedding = np.array(response.data[i].embedding)
                    cache_key = hashlib.sha256(text.encode()).hexdigest()
                    new_embeddings[cache_key] = embedding
                    
                    # Update cache
                    if CACHE_ENABLED:
                        EMBEDDING_CACHE[cache_key] = embedding
                        
            except Exception as e:
                print(f"Batch embedding failed: {e}")
                # Fallback to zero embeddings
                for text in texts_to_encode:
                    cache_key = hashlib.sha256(text.encode()).hexdigest()
                    new_embeddings[cache_key] = np.zeros(self.embedding_dim)
        
        # Combine cached and new embeddings
        result = []
        new_idx = 0
        for i, cache_key in enumerate(cache_keys):
            if cached_embeddings[i] is not None:
                result.append(cached_embeddings[i])
            else:
                result.append(new_embeddings[cache_keys[i]])
                new_idx += 1
        
        # Save cache periodically
        if CACHE_ENABLED and len(EMBEDDING_CACHE) % 50 == 0:
            save_cache(EMBEDDING_CACHE, EMBEDDING_CACHE_FILE)
                
        return result
    
    async def encode_text(self, text: str) -> np.ndarray:
        """Single text encoding (uses batch method)"""
        embeddings = await self.encode_text_batch([text])
        return embeddings[0] if embeddings else np.zeros(self.embedding_dim)
    
    def chunk_text_smartly(self, text: str, max_chunk_size: int = 500) -> List[str]:
        """Smart text chunking preserving medical context"""
        if len(text) <= max_chunk_size:
            return [text]
        
        # Split by sentences first, preserving medical terms
        sentences = re.split(r'[。！？\n]', text)
        chunks = []
        current_chunk = ""
        
        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
                
            # Check if adding this sentence would exceed limit
            test_chunk = current_chunk + "。" + sentence if current_chunk else sentence
            if len(test_chunk) > max_chunk_size and current_chunk:
                chunks.append(current_chunk)
                current_chunk = sentence
            else:
                current_chunk = test_chunk
        
        if current_chunk:
            chunks.append(current_chunk)
            
        return chunks
    
    async def build_knowledge_base(self, patient_history):
        """Optimized knowledge base building with chunking and batch processing"""
        for patient_id, records in patient_history.items():
            if not records:
                continue
                
            print(f"Building knowledge base for {patient_id}...")
            all_texts = []
            all_metadata = []
            
            for record in records:
                # Create comprehensive context text
                context_text = f"""
                Previous Visit: {record.get('summary', '')}
                Vital Signs: {record.get('vitals', '')}
                Issues: {record.get('issues', '')}
                Interventions: {record.get('interventions', '')}
                """.strip()
                
                # Smart chunking to preserve medical context
                chunks = self.chunk_text_smartly(context_text)
                
                for chunk in chunks:
                    all_texts.append(chunk)
                    all_metadata.append({
                        'date': record.get('date'),
                        'source': 'previous_visit',
                        'patient_id': patient_id,
                        'chunk_text': chunk
                    })
            
            # Batch encode all chunks for this patient
            if all_texts:
                embeddings = await self.encode_text_batch(all_texts)
                
                patient_embeddings = []
                for i, embedding in enumerate(embeddings):
                    patient_embeddings.append({
                        'text': all_metadata[i]['chunk_text'],
                        'embedding': embedding,
                        'date': all_metadata[i]['date'],
                        'source': all_metadata[i]['source']
                    })
                
                self.knowledge_base[patient_id] = patient_embeddings
                print(f"Built {len(patient_embeddings)} knowledge chunks for {patient_id}")
    
    async def retrieve_relevant_context(self, current_transcript, patient_id, top_k=3):
        """Optimized retrieval with similarity threshold and smart ranking"""
        if patient_id not in self.knowledge_base or not self.knowledge_base[patient_id]:
            return []
            
        # Encode current transcript in chunks for better matching
        transcript_chunks = self.chunk_text_smartly(current_transcript, max_chunk_size=300)
        transcript_embeddings = await self.encode_text_batch(transcript_chunks)
        
        # Calculate similarities for each transcript chunk against knowledge base
        all_similarities = []
        
        for transcript_emb in transcript_embeddings:
            if np.all(transcript_emb == 0):  # Skip zero embeddings
                continue
                
            for item in self.knowledge_base[patient_id]:
                # Reshape for cosine similarity
                transcript_emb_2d = transcript_emb.reshape(1, -1)
                item_embedding_2d = item['embedding'].reshape(1, -1)
                
                # Skip zero embeddings
                if np.all(item_embedding_2d == 0):
                    continue
                
                sim = cosine_similarity(transcript_emb_2d, item_embedding_2d)[0][0]
                
                # Apply similarity threshold
                if sim >= SIMILARITY_THRESHOLD:
                    # Add temporal decay
                    temporal_weight = self._calculate_temporal_weight(item.get('date'))
                    weighted_sim = sim * temporal_weight
                    
                    all_similarities.append((weighted_sim, item))
        
        # Remove duplicates and get top-k
        seen_texts = set()
        unique_similarities = []
        for sim, item in all_similarities:
            text_key = item['text'][:100]  # Use first 100 chars as key
            if text_key not in seen_texts:
                seen_texts.add(text_key)
                unique_similarities.append((sim, item))
        
        # Sort by similarity and return top-k
        unique_similarities.sort(reverse=True, key=lambda x: x[0])
        return [item for _, item in unique_similarities[:top_k]]
    
    def _calculate_temporal_weight(self, record_date):
        """Optimized temporal weighting"""
        if not record_date:
            return 0.8
            
        try:
            if isinstance(record_date, str):
                record_date = datetime.strptime(record_date, '%Y-%m-%d')
            days_ago = (datetime.now() - record_date).days
            # Faster exponential decay calculation
            return max(0.3, 0.95 ** (days_ago / 30))  # Decay over months, not days
        except:
            return 0.8
    
    async def generate_rag_summary(self, transcript, patient_id, categories, knowledge_text):
        """Enhanced summary with optimized RAG context"""
        # Retrieve relevant historical context
        relevant_context = await self.retrieve_relevant_context(transcript, patient_id, top_k=2)  # Reduced from 3 to 2
        
        # Build condensed context string
        context_str = ""
        if relevant_context:
            context_str = "\\n=== RELEVANT HISTORY ===\\n"
            for i, ctx in enumerate(relevant_context, 1):
                # Truncate context to save tokens
                truncated_text = ctx['text'][:300] + "..." if len(ctx['text']) > 300 else ctx['text']
                context_str += f"[{i}] {truncated_text}\\n"
        
        # Build categories (optimized format)
        cats_lines = "\\n".join(f"【{c}】" for c in categories)
        
        # Optimized prompt (reduced tokens while maintaining quality)
        rag_prompt = f"""Create professional nursing record from conversation and history.

RULES:
1. Use only conversation content and provided history
2. No speculation or creation
3. Include all dynamic categories in order
4. Be concise and factual

CONVERSATION:
{transcript}

{context_str}

BACKGROUND:
{knowledge_text}

CATEGORIES (maintain this order):
{cats_lines}

REQUIRED FORMAT:
利用者の状態（状態・トピックス）
- 【トピックス】: Critical events (falls, bleeding, fever, pain)
- Each category: Objective observations, changes, nurse responses
- <処置内容・訓練内容>: Treatment details
- <備考>: Important notes
- <外部連絡・伝達事項>: External communications
- <内部連絡・伝達事項>: Internal communications

実施した看護・リハビリテーションの内容
- 生活援助: [specific care provided]
- 排泄援助: [specific assistance]  
- 処置: [specific procedures]
- リハビリ: [specific rehabilitation]
- 指導: [specific guidance]
- 管理: [specific management]

コメント
【予定/受診】: Scheduled appointments (facility names, dates)
"""
        
        try:
            client = await self.get_client()
            response = await client.chat.completions.create(
                model=MODEL_SUMMARY,
                messages=[{"role": "user", "content": rag_prompt}],
                temperature=0.3,
                max_tokens=2000  # Limit output tokens
            )
            
            return response.choices[0].message.content, relevant_context
            
        except Exception as e:
            print(f"RAG summary generation failed: {e}")
            return "Error generating summary", []
    
# ===== Knowledge + Guard helpers =====
def strip_numbers(text: str) -> str:
    import re
    # Remove arabic + full-width digits and common numeric tokens
    text = re.sub(r"[0-9０-９％%/:\-\.]+", "", text or "")
    text = re.sub(r"\s{2,}", " ", text).strip()
    return text

def load_form_sheets(file_path: Path) -> dict[str, pd.DataFrame]:
    """Load all sheets from an Excel form into a dict."""
    xls = pd.ExcelFile(file_path)
    return {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in xls.sheet_names}

def extract_category_names(sheet_df: pd.DataFrame,
                           section_header: str = "利用者の状態（病状・トピックス）") -> list[str]:
    """
    Find the row that contains the section header, then collect tokens like 【...】 below it
    until a fully blank row.
    """
    if sheet_df is None or sheet_df.empty:
        return []

    # locate section row
    idxs = sheet_df[
        sheet_df.apply(lambda row: row.astype(str).str.contains(section_header, na=False).any(), axis=1)
    ].index

    if len(idxs) == 0:
        return []

    start = idxs[0]
    cat_pat = r"【(.*?)】"
    cats: list[str] = []

    for r in range(start + 1, sheet_df.shape[0]):
        row = sheet_df.iloc[r]
        # stop at fully blank row
        if row.isna().all():
            break
        text = " ".join(row.dropna().astype(str))
        cats.extend(re.findall(cat_pat, text))

    # normalize & dedupe (keep order)
    seen = set()
    out = []
    for c in cats:
        c = c.strip()
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    return out

def find_visiting_form_excels(root: Path) -> list[Path]:
    """Find all non-temp Excel files under root."""
    return sorted([p for p in root.rglob("*") if p.is_file()
                   and p.suffix.lower() in {".xlsx", ".xls"}
                   and not p.name.startswith("~$")])


def map_form_to_transcript_path(form_path: Path, form_root: Path, transcript_root: Path) -> Path:
    """
    Map a form path to its transcript path by matching relative path and changing extension to .txt.
      e.g. form_root/P1/abc.xlsx  -> transcript_root/P1/abc.txt
    """
    rel = Path(form_path).resolve().relative_to(Path(form_root).resolve())
    return (Path(transcript_root) / rel).with_suffix(".txt")

# =========================
# Utilities
# =========================
JP_FILLERS = r'(えー+|あー+|うー+|そのー+|えっと+|なんか|あの+|まあ|こう|そう|どう|なんていうか|なんだろう|ええと|ええ|あっ?|うっ?|んー+|ほら)'
JP_BACKCHANNELS = r'(はい+|うん+|へえ+|なるほど|わかりました|了解です|ですね)'
def clean_transcript(text: str) -> str:
    text = re.sub(r'[ \t]+', ' ', text or '')
    text = re.sub(r'[\r\n]+', '\n', text)
    text = re.sub(JP_FILLERS, '', text)
   # text = re.sub(JP_BACKCHANNELS, '', text)
    text = re.sub(r'([！？])\1+', r'\1', text)
    return text.strip()


def read_transcript_text(txt_path: Path) -> str:
    # Robust file reader (Japanese encodings)
    for enc in ("utf-8", "utf-8-sig", "cp932", "shift_jis"):
        try:
            with open(txt_path, "r", encoding=enc, errors="strict") as f:
                return f.read().replace("\ufeff", "").strip()
        except Exception:
            continue
    with open(txt_path, "r", encoding="utf-8", errors="replace") as f:
        return f.read().replace("\ufeff", "").strip()

def patient_id_from_path(p: Path) -> str:
    # assumes folder like .../P1/.../file.txt
    for part in p.parts[::-1]:
        if re.fullmatch(r"P\d+", part, re.I):
            return part.upper()
    return p.parent.name

def load_patient_profiles(csv_path: Optional[Path]) -> Dict[str,Dict[str,str]]:
    if not csv_path or not csv_path.exists():
        return {}
    df = pd.read_csv(csv_path)
    return df.set_index("patient_id").T.to_dict()

def build_patient_profile_str(patient_id: str, profiles: Dict[str,Dict[str,str]]) -> str:
    p = profiles.get(patient_id, {})
    if not p:
        return f"患者ID: {patient_id}"
    gender = str(p.get("gender",""))
    gender_ja = "女性" if gender.upper()=="F" else ("男性" if gender.upper()=="M" else gender)
    return f"患者ID: {patient_id}, 年齢: {p.get('age','')}歳, 性別: {gender_ja}, 疾患: {p.get('disease','')}"

def load_patient_history(patient_id: str, prev_dir: Path) -> List[Dict[str, Any]]:
    """Load comprehensive patient history for RAG"""
    patterns = [
        f"PrevReports - {patient_id}.csv",
        f"{patient_id}.csv", 
        f"*{patient_id}*.csv",
    ]
    
    path = None
    for pat in patterns:
        hits = list(prev_dir.glob(pat))
        if hits:
            path = hits[0]
            break
            
    if not path:
        print(f"Debug: No history file found for {patient_id}")
        return []
    
    try:
        df = pd.read_csv(path)
        print(f"Debug: Loaded CSV with columns: {df.columns.tolist()}")
        
        # Convert Field/Value format to dictionary
        history_dict = {}
        for _, row in df.iterrows():
            field = str(row.get('Field', '')).strip()
            value = row.get('Value', '')
            if field and not pd.isna(value):
                history_dict[field] = str(value).strip()
        
        # Extract main summary content
        summary_field = "利用者の状態（病状・トピックス）"
        main_summary = history_dict.get(summary_field, '')
        
        # Collect vital signs
        vital_fields = ['体温', '血圧', '脈拍', '呼吸', 'SPO2', '排便', '食欲']
        vitals_info = []
        for field in vital_fields:
            value = history_dict.get(field, '')
            if value:
                vitals_info.append(f"{field}: {value}")
        
        vitals_str = "; ".join(vitals_info) if vitals_info else ""
        
        # Create single comprehensive record
        if main_summary and len(main_summary) > 50:
            record = {
                'date': history_dict.get('Date', ''),
                'summary': main_summary,
                'vitals': vitals_str,
                'issues': '',  # Could extract from summary if needed
                'interventions': ''  # Could extract from summary if needed
            }
            print(f"Debug: Created history record with summary length: {len(main_summary)}")
            return [record]
        else:
            print(f"Debug: No substantial summary found (length: {len(main_summary)})")
            return []
                
    except Exception as e:
        print(f"Warning: Could not load history for {patient_id}: {e}")
        import traceback
        traceback.print_exc()
        return []
def load_previous_report(patient_id: str, prev_dir: Path) -> Tuple[Dict[str,str], Optional[int], Optional[str]]:
    """
    Returns (prev_report_dict, prior_recency_days, used_path)
    We accept one CSV per patient, e.g.,:
      PrevReports - P1.csv   OR   P1.csv   OR   *P1*.csv
    If a Date column exists, we compute 'recency_days' (for annotation only).
    """
    patterns = [
        f"PrevReports - {patient_id}.csv",
        f"{patient_id}.csv",
        f"*{patient_id}*.csv",
    ]
    path = None
    for pat in patterns:
        hits = list(prev_dir.glob(pat))
        if hits:
            path = hits[0]
            break
    if not path:
        return {}, None, None

    df = pd.read_csv(path)
    prev = {}
    recency_days = None
    today = date.today()
    if "Date" in df.columns:
        dates = pd.to_datetime(df["Date"], errors="coerce").dt.date.dropna()
        if len(dates):
            recency_days = (today - max(dates)).days

    for _, r in df.iterrows():
        k = str(r.get("Field","")).strip()
        v_raw = r.get("Value", "")
        v = "" if pd.isna(v_raw) else str(v_raw).strip()
        if k:
            prev[k] = v
    return prev, recency_days, str(path)

SYSTEM_SCRIBE = SystemMessagePromptTemplate.from_template("""
あなたは「臨床記録の書記（訪問看護記録用）」です。以下の会話から事実の抽出や要約のみを行います。
- 医療アドバイス・診断・治療提案は一切行いません。
- 有害行為の助長・違法行為の指南は禁止です。該当する場合は事実の有無のみを記録し。
""")

# =========================
# Stage A — Transcript-only extractor
# =========================

STAGE_A_PROMPT = """あなたは、在宅看護の会話記録から健康情報を抽出するAIアシスタントです。
会話で明示的に言及されていない情報は、一切出力しないでください。

# 出力要件
-会話内容に明示的な値があれば：必ずその値を最優先で使用してください。
- 出力は有効なJSONのみ（コードブロック・説明文・末尾カンマなし）。
- JSONキーは次の7項目のみ：体温, 血圧, 脈拍, 呼吸, SPO2, 排便, 食欲。
- 会話に値が存在しない場合、`"value"`は`null`、`"evidence"`、`"source"`は空文字(`""`)としてください。

【数値項目（体温・血圧・脈拍・SPO2）の判断ルール】
1. 信頼できる文脈でのみ抽出する: 値が「測定した」「計った」「です」「でした」などの動詞や断定の表現と共に言及されている場合を最優先する。
2. 明らかに異常な値は却下する: 体温が34°C未満または42°C以上、血圧の舒张期（下の値）が收缩期（上の値）より高い、SPO2が70%未満などの医学的に不可能・`null`とする。
3. 単独の数値に注意:
    - 血圧は「120/80」のような２つの数値の組でない場合は`null`とする。「今日が150」のような発言は収縮期のみの可能性が高く、抽出しない。
4. 範囲表現は却下する: 「95、6ぐらい」「94から上がらん」などの範囲や変化の起点となる数値は`null`とし、確定した測定値のみを抽出する。
5. その他会話中の数値を無視する: 日付（5月7日）、時間、年代、その他明らかに vital sign に関係ない数字は完全に無視する。

【排便・食欲について】
- 排便・食欲は、「有/無」を判断できる明確な発言（例：「便があった」「食べた」「食べていない」）。
- 「有/無」で出力するのは、会話で明確に言及されている場合のみです。

【排便の判断ルール】
1. 「有」とするのは「本日」の排便を直接示す発言のみ: 「出た」「あった」「便通があった」「下痢をした」
2.「無」とするのは「本日」の排便が無いと直接示す発言のみ: 「出ていない」「ない」「まだ」「便秘」
3. 以下の場合は「null」とする:
    - 「昨日出た」など、過去の分のみの言及。
    - 「便を捨ててない」など、処理に関する発言。
    - 「お通じは？」への返答がない、または「少し」「普通」などの曖昧な表現。

【食欲の判断ルール】
1. 「有」とするのは「本日」の食事を直接示す発言のみ: 「食べた」「全部食べた」「食欲がある」「完食した」
2. 「無」とするのは「本日」の食事が全く無いと直接示す発言のみ: 「食べていない」「食欲がまったくない」「一口も食べられなかった」
3. 以下の場合は「有」とする:
    - 「少し食べた」「少なかった」「あまりない」「半分食べた」など、量が少なくても摂取したことを示す発言。
4. 以下の場合は「null」とする:
    - 「昨日は食べた」など、過去の分のみの言及。
    - 「食事は？」への返答がない、または「普通」などの曖昧な表現。

#【出力形式例】
{{
  "体温":   {{"value": "36.4°C", "source": "M", "evidence": "熱は三十六度四分"}},
  "血圧":   {{"value": "124/60 mmHg", "source": "M", "evidence": "血圧が124の60"}},
  "脈拍":   {{"value": null, "source": "", "evidence": ""}},
  "呼吸":   {{"value": null, "source": "", "evidence": ""}},
  "SPO2":  {{"value": null, "source": "", "evidence": ""}},
  "排便":   {{"value": "無", "source": "M", "evidence": "この2日間は出ていない"}},
  "食欲":   {{"value": "有", "source": "M", "evidence": "ご飯は全部食べました"}}
}}

【会話内容】
{conversation}
""".strip()

stage_a_template = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template(STAGE_A_PROMPT)
])

def _empty_field():
    return {"value": "", "source": "", "evidence": ""}

def facts_today_only(stage_a_json: dict, fields: list[str]) -> dict:
    def _is_empty_val(x) -> bool:
        if x is None: return True
        s = str(x).strip().lower()
        return s in ("", "none", "null", "nan")
    out = {}
    for k in fields:
        v = stage_a_json.get(k, {}) or {}
        val_raw = v.get("value", "")
        src = (v.get("source") or "").strip()
        if src == "M" and not _is_empty_val(val_raw):
            out[k] = {
                "value": str(val_raw).strip(),
                "source": "M",
                "evidence": v.get("evidence", "") or "",
            }
        else:
            out[k] = _empty_field()
    return out

# =========================
# Safe JSON helpers
# =========================
def _safe_json_loads(s: str, context: str = "") -> Dict[str, Any]:
    """
    More robust JSON parsing with multiple fallback strategies
    """
    if not s or not isinstance(s, str):
        print(f"ERROR: Empty or invalid response for {context}")
        return {}

    original_s = s
    s = s.strip()
    
    # Strategy 1: Direct parse
    try:
        result = json.loads(s)
        return result
    except json.JSONDecodeError:
        pass

    # Strategy 2: Remove code fences
    s_clean = re.sub(r'^```(?:json)?\s*', '', s, flags=re.IGNORECASE)
    s_clean = re.sub(r'\s*```$', '', s_clean).strip()
    
    try:
        result = json.loads(s_clean)
        return result
    except json.JSONDecodeError:
        pass

    # Strategy 3: Extract JSON object
    match = re.search(r'\{[\s\S]*\}', s_clean)
    if match:
        try:
            result = json.loads(match.group())
            return result
        except json.JSONDecodeError:
            pass

    # Strategy 4: Manual repair for common issues
    try:
        # Fix trailing commas
        s_repaired = re.sub(r',\s*}', '}', s_clean)
        s_repaired = re.sub(r',\s*]', ']', s_repaired)
        # Fix missing quotes
        s_repaired = re.sub(r'(\w+):', r'"\1":', s_repaired)
        result = json.loads(s_repaired)
        return result
    except json.JSONDecodeError:
        pass

    print(f"ERROR: Failed to parse LLM response as JSON for {context}")
    print(f"Raw response: {original_s[:500]}...")
    return {}

# =========================
# Merge (Python, no overwrite; no window)
# =========================
def merge_with_prior_only_blanks(stage_a_json: Dict[str,Any],
                                 prev_report: Dict[str,str],
                                 patient_profile: Dict[str,str]) -> Dict[str,Any]:
    """
    - Only fill fields where Stage-A 'value' == "".
    - Use previous report if available -> source="P".
    - (Optional) Patient profile rarely has vitals; if you DO provide baseline fields there,
      we mark source="PP". Otherwise it is ignored.
    """
    final = json.loads(json.dumps(stage_a_json, ensure_ascii=False))

    # Accept both Japanese and common English keys in prev/profile
    keymap = {
        "体温":"体温", "血圧":"血圧", "脈拍":"脈拍", "呼吸":"呼吸", "SPO2":"SPO2", "SpO2":"SPO2",
        "排便":"排便", "食欲":"食欲",
        "BodyTemp":"体温", "BloodPressure":"血圧", "Pulse":"脈拍", "Breath":"呼吸", "Defecation":"排便", "Appetite":"食欲"
    }

    def lookup(source_dict: Dict[str,str], field_ja: str) -> Optional[str]:
        for k, v in source_dict.items():
            if keymap.get(k) == field_ja and str(v).strip():
                return str(v).strip()
        return None

    for k in FIELDS:
        v = final.get(k, {})
        curr = (isinstance(v, dict) and v.get("value", "")) or ""
        if curr:  # keep explicit mentions
            continue

        # try previous report first
        pv = lookup(prev_report, k)
        if pv:
            final[k] = {"value": pv, "source": "P", "evidence": "前回記録に基づく"}
            continue

        # optional: try patient profile *only if* you truly store such baselines there
        ppv = lookup(patient_profile, k)
        if ppv:
            final[k] = {"value": ppv, "source": "PP", "evidence": "患者プロフィールに基づく"}
        else:
            # leave empty (no I-fill in your new rules)
            final[k] = {"value": "", "source": "", "evidence": ""}

    # light normalization (e.g., blood pressure "120/70 mmHg")
    bp = final.get("血圧", {})
    if isinstance(bp, dict) and bp.get("value"):
        s = str(bp["value"]).replace("／", "/").replace(" ", "")
        # ensure "SYS/DIA mmHg"
        if "/" in s:
            s = s.replace("mmHg", "")  # strip any existing mmHg to re-add cleanly
            s = s.rstrip()
            s = f"{s} mmHg"
        bp["value"] = s

    # binary normalization
    for b in ["排便","食欲"]:
        vv = final.get(b, {})
        if isinstance(vv, dict) and vv.get("value"):
            t = vv["value"]
            if t not in ["有","無"]:
                if any(x in t for x in ["あり","出た","食べ", "摂取"]): vv["value"] = "有"
                if any(x in t for x in ["なし","出ていない","食べられない"]): vv["value"] = "無"
    return final

# ===== Optimized Knowledge generator with caching =====
async def generate_structured_knowledge_async(prev_report: dict,
                                             patient_profile: Union[Dict[str, Any], str],
                                             meter: Optional[Meter] = None) -> str:
    """Async knowledge generation with caching and optimized prompts"""
    
    # Create cache key from inputs
    profile_str = json.dumps(patient_profile, ensure_ascii=False, sort_keys=True) if isinstance(patient_profile, dict) else str(patient_profile)
    report_str = json.dumps(prev_report, ensure_ascii=False, sort_keys=True)
    cache_key = hashlib.sha256((profile_str + report_str).encode()).hexdigest()
    
    # Check cache first
    if CACHE_ENABLED and cache_key in KNOWLEDGE_CACHE:
        print("🎯 Cache HIT for knowledge generation")
        return KNOWLEDGE_CACHE[cache_key]
    
    # Optimized prompt (reduced tokens, clearer instructions)
    KNOWLEDGE_PROMPT = """Create concise background knowledge from patient data. JSON output only.

RULES:
- Summarize provided info only, NO new information
- Focus on: changes, ongoing care, important observations
- NO numbers, dates, times
- Concise bullet points

INPUT:
Profile: {patient_profile}
Previous: {prev_report}

OUTPUT JSON:
{{
    "patient_context": ["overall condition notes", "key changes/trends"],
    "care_focus": ["ongoing treatments", "symptoms to watch", "care environment"],
    "key_considerations": ["family support", "special precautions", "observation points"]
}}"""

    prompt_vars = {
        "patient_profile": profile_str,
        "prev_report": report_str,
    }
    
    try:
        client = await get_optimized_openai_client()
        
        t0 = time.time()
        response = await client.chat.completions.create(
            model=MODEL_KNOWLEDGE,
            messages=[{"role": "user", "content": KNOWLEDGE_PROMPT.format(**prompt_vars)}],
            temperature=0.0,
            response_format={"type": "json_object"}  # Force JSON
        )
        t1 = time.time()
        
        llm_response = response.choices[0].message.content
        
        if meter: 
            meter.add("Knowledge_build_async", KNOWLEDGE_PROMPT, llm_response, t1 - t0, model=MODEL_KNOWLEDGE)

        try:
            kb_json = json.loads(llm_response)
            lines = []
            
            if kb_json.get("patient_context"):
                lines.append("【患者の状況】")
                lines.extend([f"- {item}" for item in kb_json["patient_context"]])
                
            if kb_json.get("care_focus"):
                lines.append("\n【ケアの焦点】")
                lines.extend([f"- {item}" for item in kb_json["care_focus"]])
            
            if kb_json.get("key_considerations"):
                lines.append("\n【重要な考慮事項】")
                lines.extend([f"- {item}" for item in kb_json["key_considerations"]])
        
            knowledge_text = "\n".join(lines).strip()
            
            # Cache the result
            if CACHE_ENABLED:
                KNOWLEDGE_CACHE[cache_key] = knowledge_text
                # Periodically save cache
                if len(KNOWLEDGE_CACHE) % 20 == 0:
                    save_cache(KNOWLEDGE_CACHE, KNOWLEDGE_CACHE_FILE)
            
        except Exception as e:
            print(f"Knowledge JSON parsing failed: {e}")
            knowledge_text = ""
        
        return knowledge_text
        
    except Exception as e:
        print(f"Async knowledge generation failed: {e}")
        return ""


# ===== Safe summary prompt (uses knowledge_text as BACKGROUND only) =====
def build_summary_prompt_with_kb(categories: List[str]) -> ChatPromptTemplate:
    cats_lines = "\n".join(f"【{c}】" for c in categories)
    tmpl = f"""
    あなたは経験豊富な訪問看護師です。以下の情報をもとに、専門的で正確な看護記録を作成してください。

【絶対規則】
1. 会話内容と知識情報のみを根拠とする
2. 推測や創作は一切禁止
3. 数値・日付・時間は会話に明記されている場合のみ記載
4. 動的カテゴリは指定された順序で必ずすべて記載
5. 「実施した看護・リハビリテーションの内容」セクションには、会話の内容に基づいて看護師が患者に対して行ったすべてのケアを、同一または類似の名称で記載してください。例: バイタルチェック、状態観察、全身清拭、オムツ交換、パット交換、創処置、褥瘡処置、キャビロン塗布、呼吸訓練、介護指導、ＡＤＬ指導、療養指導、ストマ管理、薬物管理、在宅酸素管理。
6. 全体を簡潔に、要点をまとめて記述すること。冗長な表現は避ける。

—参考例—
「参考例は、体裁、記述スタイル、具体性の手本です。参考例の具体的な内容（人名や数値など）を出力にそのまま含めてはいけません。」
                                             
利用者の状態（状態・トピックス）
    体調変化なし。パーキンソン症状としては、動きづらさ（特に朝）が感じている。ベットへ着席前には、電話がなり、気持ちが先立って倒れ込む様子あり、スタッフ保持していたため、転倒には至らず。  
    【体調】風邪症状なし。右大腿痛増強あるが、カロナール内服は無し。昼食は餅を摂取し、胃部不快はなし。
    【褥瘡他】（右大転子部）コアグラ極少量。クリーム色アイテル中等量。（臀部）クリーム色アイテル中等量。
    【ストマ】便貯留（性3-4、量2－3)あり。ストマ３時方向に軽度糜爛あり。周囲発赤あるも自覚症状なし。
    <処置内容・訓練内容>
    【褥瘡処置】（右大転子部）イソジンゲル＋不織布＋パット。周囲皮膚はワセリン+アズノール塗布。
    【ストマ管理】洗浄後にパウチ交換。３時方向にアクトシン塗布、近接部にパウダー塗布。
    【保湿剤塗布】全身にヘパリンフォーム、乾燥部（両腰部）にワセリン塗布追加。
    <備考>
    綿棒お渡し済み、代金預かりあり。
    <外部連絡・伝達事項>
    MCSで報告予定。
    <内部連絡・伝達事項>

実施した看護・リハビリテーションの内容
    生活援助:バイタルチェック、状態観察、全身清拭、ひげそり、爪切り、水分摂取介助、精神的援助、介護者支援
    排泄援助:パウチ交換、オムツ交換、パット交換	
    処置:外用薬塗布、創処置、褥瘡処置、キャビロン塗布
    リハビリ:呼吸訓練
    指導:観察方法指導、合併症予防指導、排便調整指導、食事指導、服薬指導、社会資源活用指導、緊急時対応指導、在宅酸素療法指導、介護指導、ＡＤＬ指導、療養指導、水分摂取指導
    管理:ストマ管理、薬物管理、在宅酸素管理

コメント:
    【受診について】: 2/4 呼吸器科、2/5 眼科の受診予定あり。

# 情報の優先順位（厳守）:
1. 【Transcript】が最優先 - 会話の実際の内容を最も重視
2. 【Knowledge】は背景参考 - 患者の経過や傾向の理解にのみ使用

【Knowledge】（背景情報）
{{knowledge_text}}

【動的カテゴリ】(この順序と見出しを守る)
{cats_lines}

【Transcript】（会話全文）
{{transcript}}

# 最終要約形式:
以下の形式を参考に、入力情報に基づいて独自の要約を作成してください。

利用者の状態（状態・トピックス）
各サブカテゴリ (例: [身体状態]、[褥瘡]、[ストーマ] など) について、以下のように説明してください:
- 【トピックス】には、転倒、出血、発熱、強い痛みなど、患者の安全に関わる最も重要な出来事を必ず要約してください。
- {cats_lines}: 各カテゴリーについて、観察された事実を客観的かつ簡潔に記述してください。症状、行動、身体的・精神的状態をより深く掘り下げて、観察された変化や看護師の対応、推測される次のステップなども含めてください。症状の進行や、文脈から推測される問題点に関しても推論を行ってください。
- <処置内容・訓練内容>: 関連する治療の詳細やトレーニングの詳細を書く。
- <備考>: 特別または重要な備考を書く。
- <外部連絡・伝達事項>: 外部機関や利害関係者の連絡先を書く。
- <内部連絡・伝達事項>: 内部コミュニケーションのための連絡先を記入する。

実施した看護/リハビリテーションの内容
※記録に基づいて、看護師が提供したすべてのケアの名前を書きます。
    - 生活援助:[具体的な実施内容]
    - 排泄援助:[具体的な実施内容]
    - 処置:[具体的な実施内容]
    - リハビリ:[具体的な実施内容]
    - 指導:[具体的な実施内容]
    - 管理:[具体的な実施内容]

コメント
    【予定/受診】会話に基づく予定のみ（施設名・日付など） 
""".strip()
    return ChatPromptTemplate.from_messages([SYSTEM_SCRIBE,HumanMessagePromptTemplate.from_template(tmpl)])

# =========================
# Rendering helper for metering
# =========================
def _render_messages_text(prompt_tmpl: ChatPromptTemplate, variables: Dict[str, Any]) -> Tuple[str, List[Any]]:
    """
    Render a ChatPromptTemplate to messages and return a single prompt_text for metering.
    Avoids using .template.format(...) which breaks with curly braces inside JSON.
    """
    msgs = prompt_tmpl.format_messages(**variables)
    prompt_text = "\n\n".join(getattr(m, "content", str(m)) for m in msgs)
    return prompt_text, msgs


# =========================
# Core: run one transcript (3 simple steps)
# =========================
def run_one_transcript(
    transcript_path: Path,
    prevreports_dir: Optional[Path] = PREVREPORTS_DIR,
    profiles_csv: Optional[Path] = PATIENT_PROFILE_CSV,
    categories: Optional[List[str]] = None,
) -> Dict[str, Any]:

    transcript_path = Path(transcript_path)
    patient_id = patient_id_from_path(transcript_path)

    try:
        # Load artifacts
        full_transcript = clean_transcript(read_transcript_text(transcript_path))
        prev_report, _, prev_path = load_previous_report(patient_id, Path(prevreports_dir))
        profiles = load_patient_profiles(Path(profiles_csv))
        patient_profile_dict = profiles.get(patient_id, {})  # dict for merge

        meter = Meter()

        # Step 1: Extraction (strict)
        print(f"\n=== Processing {patient_id} ===")
        t0 = time.time()
        msgs = stage_a_template.format_messages(conversation=full_transcript)
        print("Sending prompt to LLM...")
        resp = llm_extract.invoke(msgs)
        t1 = time.time()
        stage_a_json_str = resp.content
        print(f"LLM Response: '{stage_a_json_str}...'")  # Show first 100 chars
        
        meter.add("StageA_extract", "\n\n".join(m.content for m in msgs), stage_a_json_str, t1 - t0, model=MODEL_EXTRACT)
        
        # Use improved parser with context for debugging
        stage_a_json = _safe_json_loads(stage_a_json_str, context=f"patient {patient_id}")
        
        # If parsing failed completely, initialize an empty structure to avoid crashes later
        if not stage_a_json:
            stage_a_json = {field: {"value": "", "source": "", "evidence": ""} for field in FIELDS}
            print(f"WARNING: Using empty facts for {patient_id} due to JSON parse failure.")

        # Step 2: Merge blanks with priors
        final_today = facts_today_only(stage_a_json, FIELDS) 
        #final_today = postprocess_finals(final_today)
        #final_filled = merge_with_prior_only_blanks(stage_a_json, prev_report, patient_profile_dict)  # reference-only

        # Step 3: Summarization (transcript + Final_Facts only)
        # Step 3a: Generate background knowledge (non-numeric) from profile + prior
        # Categories: use those passed in; if empty, fall back to a reasonable default
        
        knowledge_text = generate_structured_knowledge(
            prev_report=prev_report,
            patient_profile=patient_profile_dict or build_patient_profile_str(patient_id, profiles),
            meter=meter
        )
        print("knowledge_text", knowledge_text)
        # Step 3b: Safe summarization (Transcript + Final_Facts + Knowledge)
        cats = inject_topics_category(categories, full_transcript)
        summary_tmpl = build_summary_prompt_with_kb(cats)
        #summary_tmpl = build_summary_prompt_with_kb(categories or [])
        sum_vars = {
            "final_facts": json.dumps(final_today, ensure_ascii=False),
            "knowledge_text": knowledge_text,
            "transcript": full_transcript
        }
        sum_prompt_text, sum_msgs = _render_messages_text(summary_tmpl, sum_vars)

        t0 = time.time()
        summary_out = llm_summarize.invoke(sum_msgs).content
        t1 = time.time()
        meter.add("Summarize_safe", sum_prompt_text, summary_out, t1 - t0, model=MODEL_SUMMARY)
  
        return {
            "patient_id": patient_id,
            "transcript_path": str(transcript_path),
            "previous_report_source": prev_path,
            "Transcript": full_transcript,
            "StageA_JSON": json.dumps(stage_a_json, ensure_ascii=False),
            "Final_Facts": json.dumps(final_today, ensure_ascii=False),
            "Knowledge_Text": knowledge_text,
            "Generated Summary": summary_out,
            "timing_token_cost_table": meter.as_dataframe().to_dict(orient="records"),
            "timing_token_cost_totals": meter.totals(),
        }

    
    except Exception as e:
        print(f"CRITICAL ERROR in run_one_transcript for {transcript_path}: {str(e)}")
        print(f"Error type: {type(e).__name__}")
        import traceback
        traceback.print_exc()
        # Return a minimal result to keep the batch going
        return {
            "patient_id": patient_id,
            "transcript_path": str(transcript_path),
            "error": str(e),
            "StageA_JSON": "{}",
            "Final_Facts": "{}",
            "Knowledge_JSON": "{}",
            "Knowledge_Text": "",
            "Generated Summary": "ERROR: Failed to process"
        }

async def run_one_transcript_optimized(
    transcript_path: Path,
    prevreports_dir: Optional[Path] = PREVREPORTS_DIR,
    profiles_csv: Optional[Path] = PATIENT_PROFILE_CSV,
    categories: Optional[List[str]] = None,
    rag_system: Optional[OptimizedClinicalRAG] = None
) -> Dict[str, Any]:
    """Optimized single transcript processing with async operations"""
    
    transcript_path = Path(transcript_path)
    patient_id = patient_id_from_path(transcript_path)

    try:
        # Load artifacts (I/O can be optimized but keeping simple for now)
        full_transcript = clean_transcript(read_transcript_text(transcript_path))
        prev_report, _, prev_path = load_previous_report(patient_id, Path(prevreports_dir))
        profiles = load_patient_profiles(Path(profiles_csv))
        patient_profile_dict = profiles.get(patient_id, {})
        
        meter = Meter()

        # Step 1: Async extraction with optimized JSON mode
        print(f"\\n=== Processing {patient_id} ===")
        stage_a_json = await extract_with_json_mode(full_transcript, patient_id)
        
        # Step 2: Merge blanks with priors (same logic, kept fast)
        final_today = facts_today_only(stage_a_json, FIELDS) 

        # Step 3: Async knowledge generation with caching
        knowledge_text = await generate_structured_knowledge_async(
            prev_report=prev_report,
            patient_profile=patient_profile_dict or build_patient_profile_str(patient_id, profiles),
            meter=meter
        )
        print("Knowledge generated:", len(knowledge_text), "chars")

        # Step 4: RAG-Enhanced or Standard Summarization
        cats = inject_topics_category(categories, full_transcript)
        
        if rag_system:
            # Build knowledge base for this patient (with caching in RAG system)
            patient_history = load_patient_history(patient_id, Path(prevreports_dir))
            await rag_system.build_knowledge_base({patient_id: patient_history})
            
            # Generate RAG-enhanced summary
            t0 = time.time()
            summary_out, rag_context = await rag_system.generate_rag_summary(
                full_transcript, patient_id, cats, knowledge_text
            )
            t1 = time.time()
            
            meter.add("Summarize_RAG", f"RAG Summary for {patient_id}", summary_out, t1 - t0, model=MODEL_SUMMARY)
            print(f"RAG: Used {len(rag_context)} historical context items")
            
        else:
            # Fallback to standard summarization
            summary_tmpl = build_summary_prompt_with_kb(cats)
            sum_vars = {
                "final_facts": json.dumps(final_today, ensure_ascii=False),
                "knowledge_text": knowledge_text,
                "transcript": full_transcript
            }
            sum_prompt_text, sum_msgs = _render_messages_text(summary_tmpl, sum_vars)

            t0 = time.time()
            summary_out = llm_summarize.invoke(sum_msgs).content
            t1 = time.time()
            meter.add("Summarize_standard", sum_prompt_text, summary_out, t1 - t0, model=MODEL_SUMMARY)
            rag_context = []

        return {
            "patient_id": patient_id,
            "transcript_path": str(transcript_path),
            "previous_report_source": prev_path,
            "Transcript": full_transcript,
            "StageA_JSON": json.dumps(stage_a_json, ensure_ascii=False),
            "Final_Facts": json.dumps(final_today, ensure_ascii=False),
            "Knowledge_Text": knowledge_text,
            "Generated_Summary": summary_out,
            "RAG_Context_Used": len(rag_context) if rag_system else 0,
            "timing_token_cost_table": meter.as_dataframe().to_dict(orient="records"),
            "timing_token_cost_totals": meter.totals(),
        }

    except Exception as e:
        print(f"ERROR in run_one_transcript_optimized for {transcript_path}: {str(e)}")
        import traceback
        traceback.print_exc()
        return {
            "patient_id": patient_id,
            "transcript_path": str(transcript_path),
            "error": str(e),
            "StageA_JSON": "{}",
            "Final_Facts": "{}",
            "Knowledge_Text": "",
            "Generated_Summary": "ERROR: Failed to process",
            "RAG_Context_Used": 0
        }

async def run_one_form_optimized(
    form_path: Path,
    form_root: Path,
    transcript_root: Path,
    prevreports_dir: Path = PREVREPORTS_DIR,
    profiles_csv: Path = PATIENT_PROFILE_CSV,
    rag_system: Optional[OptimizedClinicalRAG] = None
) -> dict:
    """Optimized form processing with category extraction and async pipeline"""
    # 1) Extract categories from form
    sheets = load_form_sheets(form_path)
    sheet1 = sheets.get("Sheet1")
    cats = extract_category_names(sheet1, "利用者の状態（病状・トピックス）")

    # 2) Find corresponding transcript
    tpath = map_form_to_transcript_path(form_path, form_root, transcript_root)
    if not tpath.exists():
        raise FileNotFoundError(f"Transcript not found for form: {form_path} -> {tpath}")

    # 3) Use optimized async pipeline
    return await run_one_transcript_optimized(
        transcript_path=tpath,
        prevreports_dir=prevreports_dir,
        profiles_csv=profiles_csv,
        categories=cats,
        rag_system=rag_system
    )
# =========================
# Writer: compact CSV row for the 7 items
# =========================
def final_facts_to_row(file_name: str, final_facts: Dict[str,Any]) -> Dict[str,str]:
    row = {"File name": file_name}
    for k in FIELDS:
        v = final_facts.get(k, {})
        row[k] = v.get("value","")
        row[f"{k}【出典】"] = v.get("source","")
        row[f"{k}根拠"]    = (v.get("evidence","") or "")[:60]
    return row

def inject_topics_category(categories: list[str], transcript: str) -> list[str]:
    cats = list(categories) if categories else []
    if "トピックス" in cats:
        return cats

    # --- toggle to force insertion regardless of cues ---
    ALWAYS_INSERT = False
    if ALWAYS_INSERT:
        cats.insert(0, "トピックス")
        return cats

    # Normalize transcript (NFKC catches ＰＤ→PD など)
    t = unicodedata.normalize("NFKC", transcript or "")

    # Minimal, conservative cue set (expand as needed)
    hot_patterns = [
        r"転倒",                                   # falls
        r"(夜間|就寝).*(目が覚め|中途覚醒|頻尿)",      # night awakenings/urination
        r"(受診|外来|予約|検査|紹介)",                 # visits/tests/referrals
        r"(入院|退院|転院|救急)",                     # admissions/discharges/ER
        r"(施設|入居|PD\s*ハウス|ショート(?:ステイ)?)", # facility / PDハウス / short-stay
        r"(薬|内服|処方|変更|増減|飲み忘れ|貼付|パッチ)", # med changes/adherence/patch
        r"(発熱|発作|疼痛|痛み|出血|呼吸苦|SpO?2|酸素)", # acute symptoms / O2
        r"(褥瘡|創|皮疹|やけど)",                     # skin/wound
        r"(誤嚥|嘔吐|下痢)",                          # GI red flags
    ]

    need_topics = any(re.search(p, t) for p in hot_patterns)
    if need_topics:
        cats.insert(0, "トピックス")
    return cats

# =========================
# Post-extraction validators
# =========================

def normalize_bp(s: str) -> Optional[str]:
    if not s: return None
    s = s.replace('／','/').replace(' ', '')
    m = re.search(r'(\d{2,3})\s*/\s*(\d{2,3})', s)
    if not m: return None
    sys, dia = int(m.group(1)), int(m.group(2))
    if not (60 <= sys <= 250 and 30 <= dia <= 150): return None
    return f"{sys}/{dia} mmHg"

def normalize_temp(s: str) -> Optional[str]:
    if not s: return None
    # 36.0 / 36度4分 / 36度4
    s = s.replace('℃','°C')
    m = re.search(r'(\d{2})(?:[度°]\s*(\d))?(?:[.,](\d))?', s)
    if not m: return None
    a, b, c = m.group(1), m.group(2), m.group(3)
    if b and not c:  # 36度4(分) → 36.4
        val = float(f"{a}.{b}")
    elif c:          # 36,4 / 36.4
        val = float(f"{a}.{c}")
    else:
        val = float(a)
    if not (33.0 <= val <= 42.5): return None
    return f"{val:.1f}°C"

def normalize_rate(s: str, unit: str) -> Optional[str]:
    if not s: return None
    m = re.search(r'(\d{1,3})', s)
    if not m: return None
    v = int(m.group(1))
    ok = (20 <= v <= 200) if unit == "bpm" else (6 <= v <= 40) # 呼吸
    if not ok: return None
    return f"{v}回/分" if unit=="rpm" else f"{v}回/分"  # both表記合わせ

def normalize_spo2(s: str) -> Optional[str]:
    if not s: return None
    m = re.search(r'(\d{2,3})\s*%?', s)
    if not m: return None
    v = int(m.group(1))
    if not (50 <= v <= 100): return None
    return f"{v}%"

def normalize_binary(s: str) -> Optional[str]:
    if not s: return None
    s2 = s.strip()
    if any(x in s2 for x in ["有","あり","出た","食べ", "摂取"]): return "有"
    if any(x in s2 for x in ["無","なし","出ていない","食べられない"]): return "無"
    return None

def postprocess_finals(final: dict) -> dict:
    out = json.loads(json.dumps(final, ensure_ascii=False))
    def _set(k, val):
        if val:
            out[k]["value"] = val
    _set("血圧", normalize_bp(out["血圧"]["value"]))
    _set("体温", normalize_temp(out["体温"]["value"]))
    _set("脈拍", normalize_rate(out["脈拍"]["value"], "bpm"))
    _set("呼吸", normalize_rate(out["呼吸"]["value"], "rpm"))
    _set("SPO2", normalize_spo2(out["SPO2"]["value"]))
    for k in ["排便","食欲"]:
        _set(k, normalize_binary(out[k]["value"]))
    return out


# =========================
# Batch over all transcripts
# =========================

def run_one_form(
    form_path: Path,
    form_root: Path,
    transcript_root: Path,
    prevreports_dir: Path = PREVREPORTS_DIR,
    profiles_csv: Path = PATIENT_PROFILE_CSV,
) -> dict:
    # 1) Extract categories from form
    sheets = load_form_sheets(form_path)
    sheet1 = sheets.get("Sheet1")

    cats = extract_category_names(sheet1, "利用者の状態（病状・トピックス）")

    
    print("cats", cats)

    # 2) Find corresponding transcript
    tpath = map_form_to_transcript_path(form_path, form_root, transcript_root)
    if not tpath.exists():
        raise FileNotFoundError(f"Transcript not found for form: {form_path} -> {tpath}")

    # 3) Reuse your simple pipeline, but pass categories so summary aligns to the form
    return run_one_transcript(
        transcript_path=tpath,
        prevreports_dir=prevreports_dir,
        profiles_csv=profiles_csv,
        categories=cats,
    )

def discover_transcripts(root: Path) -> List[Path]:
    return sorted([p for p in root.rglob("*.txt") if p.is_file()])

class OpenAIClinicalRAG:
    def __init__(self, api_key):
        import openai
        self.client = openai.AsyncOpenAI(api_key=api_key)
        self.embedding_model = "text-embedding-3-small"
        self.knowledge_base = {}
        
    async def encode_text(self, text):
        """Get embeddings using OpenAI API"""
        if not text or text.strip() == "":
            return np.zeros(1536)  # Default embedding dimension
            
        response = await self.client.embeddings.create(
            input=text,
            model=self.embedding_model
        )
        return np.array(response.data[0].embedding)
    
    async def build_knowledge_base(self, patient_history):
        """Build searchable knowledge base from patient history"""
        for patient_id, records in patient_history.items():
            if not records:
                continue
                
            patient_embeddings = []
            
            for record in records:
                # Combine relevant fields into searchable text
                context_text = f"""
                Previous Visit Notes: {record.get('summary', '')}
                Vital Signs: {record.get('vitals', '')}
                Key Issues: {record.get('issues', '')}
                Interventions: {record.get('interventions', '')}
                """
                
                embedding = await self.encode_text(context_text)
                patient_embeddings.append({
                    'text': context_text,
                    'embedding': embedding,
                    'date': record.get('date'),
                    'source': 'previous_visit'
                })
            
            self.knowledge_base[patient_id] = patient_embeddings
    
    async def retrieve_relevant_context(self, current_transcript, patient_id, top_k=3):
        """Retrieve most relevant historical context"""
        if patient_id not in self.knowledge_base or not self.knowledge_base[patient_id]:
            return []
            
        # Encode current transcript
        current_embedding = await self.encode_text(current_transcript)
        
        # Calculate similarities
        similarities = []
        for item in self.knowledge_base[patient_id]:
            # Reshape for cosine similarity
            current_embedding_2d = current_embedding.reshape(1, -1)
            item_embedding_2d = item['embedding'].reshape(1, -1)
            
            sim = cosine_similarity(current_embedding_2d, item_embedding_2d)[0][0]
            
            # Add temporal decay (prefer recent records)
            temporal_weight = self._calculate_temporal_weight(item.get('date'))
            weighted_sim = sim * temporal_weight
            
            similarities.append((weighted_sim, item))
        
        # Return top-k most relevant
        similarities.sort(reverse=True, key=lambda x: x[0])
        return [item for _, item in similarities[:top_k]]
    
    def _calculate_temporal_weight(self, record_date):
        """Weight recent records higher"""
        if not record_date:
            return 0.8  # Default weight for undated records
            
        try:
            if isinstance(record_date, str):
                record_date = datetime.strptime(record_date, '%Y-%m-%d')
            days_ago = (datetime.now() - record_date).days
            # Exponential decay: recent (1.0) to 90+ days ago (0.3)
            return max(0.3, np.exp(-days_ago / 60))
        except:
            return 0.8
    
    async def generate_rag_summary(self, transcript, patient_id, categories, knowledge_text):
        """Enhanced summary with RAG context"""
        # Retrieve relevant historical context
        relevant_context = await self.retrieve_relevant_context(transcript, patient_id)
        
        # Build context string
        context_str = ""
        if relevant_context:
            context_str = "\\n=== RELEVANT PATIENT HISTORY ===\\n"
            for i, ctx in enumerate(relevant_context, 1):
                context_str += f"\\n--- Historical Context {i} ---\\n{ctx['text']}\\n"
        
        # Build categories lines
        cats_lines = "\\n".join(f"【{c}】" for c in categories)
        
        # Enhanced prompt with RAG context and complete formatting
        rag_prompt = f"""
        あなたは経験豊富な訪問看護師です。以下の情報をもとに、専門的で正確な看護記録を作成してください。

        【絶対規則】
        1. 会話内容、履歴情報、知識情報のみを根拠とする
        2. 推測や創作は一切禁止
        3. 数値・日付・時間は会話に明記されている場合のみ記載
        4. 動的カテゴリは指定された順序で必ずすべて記載
        5. 「実施した看護・リハビリテーションの内容」セクションには、会話の内容に基づいて看護師が患者に対して行ったすべてのケアを、同一または類似の名称で記載してください。
        6. 全体を簡潔に、要点をまとめて記述すること。冗長な表現は避ける。

        【RAG重要ガイドライン】
        1. 履歴情報を使用してパターン、変化、傾向を特定する
        2. 関連する場合は現在の状態と過去の訪問を比較する
        3. 改善、悪化、または繰り返される問題に注意する
        4. すべての情報源から厳格な事実の正確性を維持する
        5. 履歴情報が現在の記録と矛盾する場合は、現在の記録を優先するが相違点を記載する

        【現在の訪問記録】
        {transcript}

        {context_str}

        【背景知識】
        {knowledge_text}

        【動的カテゴリ】(この順序と見出しを守る)
        {cats_lines}

        # 情報の優先順位（厳守）:
        1. 【現在の記録】が最優先 - 今回の会話の実際の内容を最も重視
        2. 【履歴情報】は変化・傾向の把握に使用 - パターンや経過の理解
        3. 【背景知識】は参考情報 - 患者の全体的な理解にのみ使用

        # 最終要約形式:
        以下の形式を参考に、入力情報に基づいて独自の要約を作成してください。

        利用者の状態（状態・トピックス）
        各サブカテゴリについて、以下のように説明してください:
        - 【トピックス】には、転倒、出血、発熱、強い痛みなど、患者の安全に関わる最も重要な出来事を必ず要約してください。履歴との比較も含める。
        - {cats_lines}: 各カテゴリーについて、観察された事実を客観的かつ簡潔に記述してください。症状、行動、身体的・精神的状態をより深く掘り下げて、観察された変化や看護師の対応、推測される次のステップなども含めてください。履歴情報がある場合は経過や変化も記載。
        - <処置内容・訓練内容>: 関連する治療の詳細やトレーニングの詳細を書く。
        - <備考>: 特別または重要な備考を書く。
        - <外部連絡・伝達事項>: 外部機関や利害関係者の連絡先を書く。
        - <内部連絡・伝達事項>: 内部コミュニケーションのための連絡先を記入する。

        実施した看護・リハビリテーションの内容
        ※記録に基づいて、看護師が提供したすべてのケアの名前を書きます。
            - 生活援助:[具体的な実施内容]
            - 排泄援助:[具体的な実施内容]
            - 処置:[具体的な実施内容]
            - リハビリ:[具体的な実施内容]
            - 指導:[具体的な実施内容]
            - 管理:[具体的な実施内容]

        コメント
            【予定/受診】会話に基づく予定のみ（施設名・日付など）
        """
        
        response = await self.client.chat.completions.create(
            model=MODEL_SUMMARY,
            messages=[{"role": "user", "content": rag_prompt}],
            temperature=0.3
        )
        
        return response.choices[0].message.content, relevant_context

async def run_one_form_with_rag(
    form_path: Path,
    form_root: Path,
    transcript_root: Path,
    prevreports_dir: Path = PREVREPORTS_DIR,
    profiles_csv: Path = PATIENT_PROFILE_CSV,
    rag_system: Optional[OpenAIClinicalRAG] = None
) -> dict:
    # 1) Extract categories from form
    sheets = load_form_sheets(form_path)
    sheet1 = sheets.get("Sheet1")
    cats = extract_category_names(sheet1, "利用者の状態（病状・トピックス）")
    print("cats", cats)

    # 2) Find corresponding transcript
    tpath = map_form_to_transcript_path(form_path, form_root, transcript_root)
    if not tpath.exists():
        raise FileNotFoundError(f"Transcript not found for form: {form_path} -> {tpath}")

    # 3) Use RAG-enhanced pipeline
    return await run_one_transcript_with_rag(
        transcript_path=tpath,
        prevreports_dir=prevreports_dir,
        profiles_csv=profiles_csv,
        categories=cats,
        rag_system=rag_system
    )

async def main():
    # Initialize RAG system
    rag_system = OpenAIClinicalRAG(api_key=OPENAI_API_KEY)
    
    forms = find_visiting_form_excels(FORM_ROOT)
    print(f"Discovered {len(forms)} forms under {FORM_ROOT}")

    batch_results = []
    rows7 = []
    rows7_prior = []

    grand_cost = grand_in = grand_out = 0
    grand_time = 0.0

    for fpath in tqdm(forms, desc="Processing Forms with RAG", unit="form"):
        try:
            # Update run_one_form to be async
            res = await run_one_form_with_rag(
                form_path=fpath,
                form_root=FORM_ROOT,
                transcript_root=TRANSCRIPT_ROOT,
                prevreports_dir=PREVREPORTS_DIR,
                profiles_csv=PATIENT_PROFILE_CSV,
                rag_system=rag_system
            )

            batch_results.append(res)
            print("transcript_path: ", res["transcript_path"])
            print("Knowledge_Text", res["Knowledge_Text"][:100] + "..." if len(res["Knowledge_Text"]) > 100 else res["Knowledge_Text"])
            print("RAG Context Used:", res.get("RAG_Context_Used", 0))
            print("Generated Summary", res["Generated_Summary"][:200] + "..." if len(res["Generated_Summary"]) > 200 else res["Generated_Summary"])

            # Save per-form workbook
            out_xlsx = OUT_DIR / f"summarized_transcript_RAG_{Path(res['transcript_path']).stem}.xlsx"
            pd.DataFrame([res]).to_excel(out_xlsx, index=False)

            # Append compact 7-field row
            ff_today = json.loads(res["Final_Facts"])
            ff_prior = json.loads(res["Final_Facts_filled_from_prior"])

            rows7.append(final_facts_to_row(Path(res["transcript_path"]).name, ff_today))
            rows7_prior.append(final_facts_to_row(Path(res["transcript_path"]).name, ff_prior))

            # Totals
            totals = res["timing_token_cost_totals"]
            grand_in += totals["input_tokens"]
            grand_out += totals["output_tokens"]
            grand_cost += totals["cost_usd"]
            grand_time += totals["seconds"]

        except Exception as e:
            print(f"[ERR] {fpath}: {e}")

    # Save batch tables
    pd.DataFrame(batch_results).to_excel(OUT_DIR / "summarized_from_forms_RAG.xlsx", index=False)
    pd.DataFrame(rows7).to_csv(OUT_DIR / "extracted_health_fields_RAG.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(rows7_prior).to_csv(OUT_DIR / "extracted_health_fields_RAG_prior.csv", index=False, encoding="utf-8-sig")

    print("\\n=== GRAND TOTALS ===")
    print(f"Input tokens: {grand_in}")
    print(f"Output tokens: {grand_out}")
    print(f"Est. cost: ${grand_cost:.4f}")
    print(f"Wall time: {grand_time:.2f}s")

# For Jupyter notebooks, run the async function directly
await main()

